In [2]:
import os
import tarfile
import pandas as pd
from google.colab import drive
import GEOparse
import urllib.request

if os.path.ismount('/content/drive'):
    drive.flush_and_unmount()
    if os.path.ismount('/content/drive'):
        !fusermount -uz /content/drive

if os.path.exists('/content/drive') and os.path.isdir('/content/drive') and len(os.listdir('/content/drive')) > 0:
    !rm -r /content/drive/*

os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive', force_remount=True)

PROJECT_PATH = '/content/drive/MyDrive/PD ISEF project'
RAW_DIR = os.path.join(PROJECT_PATH, 'data/raw/')
PROCESSED_DIR = os.path.join(PROJECT_PATH, 'data/processed/')

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("[+] Workspace locked in and folders verified!")

Mounted at /content/drive
[+] Workspace locked in and folders verified!


In [3]:
PROJECT_PATH = '/content/drive/MyDrive/PD ISEF project'
RAW_DIR = os.path.join(PROJECT_PATH, 'data/raw/')
os.makedirs(RAW_DIR, exist_ok=True)

print("[-] Opening secure API bridge to NCBI GEO...")
print("[-] Attempting programmatic download of Master Series: GSE269781")

try:
    gse = GEOparse.get_GEO(geo="GSE269781", destdir=RAW_DIR, silent=False)

    print("\n[+] Extraction Engine Connected Successfully!")
    print(f"-> Title: {gse.metadata['title'][0]}")
    print(f"-> Total Samples Identified: {len(gse.gsms)}")

    sample_ids = list(gse.gsms.keys())
    print("\n[-] Verifying sample tracking registry:")
    for sid in sample_ids[:3]:
        print(f"   Sample ID: {sid} | Title: {gse.gsms[sid].metadata['title'][0]}")

except Exception as e:
    print(f"\n[X] Connection/Download Error: {e}")
    print("If this fails, NCBI's FTP server might be rate-limiting. Double check your internet stability.")

10-Jun-2026 18:26:00 DEBUG utils - Directory /content/drive/MyDrive/PD ISEF project/data/raw/ already exists. Skipping.
DEBUG:GEOparse:Directory /content/drive/MyDrive/PD ISEF project/data/raw/ already exists. Skipping.
10-Jun-2026 18:26:00 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
10-Jun-2026 18:26:00 INFO GEOparse - Parsing /content/drive/MyDrive/PD ISEF project/data/raw/GSE269781_family.soft.gz: 
INFO:GEOparse:Parsing /content/drive/MyDrive/PD ISEF project/data/raw/GSE269781_family.soft.gz: 


[-] Opening secure API bridge to NCBI GEO...
[-] Attempting programmatic download of Master Series: GSE269781


10-Jun-2026 18:26:01 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
10-Jun-2026 18:26:01 DEBUG GEOparse - SERIES: GSE269781
DEBUG:GEOparse:SERIES: GSE269781
10-Jun-2026 18:26:01 DEBUG GEOparse - PLATFORM: GPL17301
DEBUG:GEOparse:PLATFORM: GPL17301
10-Jun-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM8326771
DEBUG:GEOparse:SAMPLE: GSM8326771
10-Jun-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM8326772
DEBUG:GEOparse:SAMPLE: GSM8326772
10-Jun-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM8326773
DEBUG:GEOparse:SAMPLE: GSM8326773
10-Jun-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM8326774
DEBUG:GEOparse:SAMPLE: GSM8326774
10-Jun-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM8326775
DEBUG:GEOparse:SAMPLE: GSM8326775
10-Jun-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM8326776
DEBUG:GEOparse:SAMPLE: GSM8326776
10-Jun-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM8326777
DEBUG:GEOparse:SAMPLE: GSM8326777
10-Jun-2026 18:26:01 DEBUG GEOparse - SAMPLE: GSM8326778
DEBUG:GEOparse:SAMPLE: GSM8326778
1


[+] Extraction Engine Connected Successfully!
-> Title: Comprehensive data for studying serum exosome microRNA transcriptome in Parkinson's disease patients
-> Total Samples Identified: 397

[-] Verifying sample tracking registry:
   Sample ID: GSM8326771 | Title: Serum exosome miRNA, PD, 2020, 01
   Sample ID: GSM8326772 | Title: Serum exosome miRNA, PD, 2020, 02
   Sample ID: GSM8326773 | Title: Serum exosome miRNA, PD, 2020, 03


In [4]:
print("[-] Parsing GEO dataset object from repository...")
gse_object = GEOparse.get_GEO(geo="GSE269775", destdir=RAW_DIR)

sample_list = []
disease_list = []
time_point_list = []
characteristics_list = []

print("[-] Extracting clinical attributes from parsed GSM objects...")

for gsm_id, gsm_obj in gse_object.gsms.items():
    characteristics = gsm_obj.metadata.get('characteristics_ch1', [])
    char_string = " | ".join(characteristics).lower()

    if 'pd' in char_string or "parkinson" in char_string:
        disease = "Parkinson's Disease"
    elif 'control' in char_string or 'healthy' in char_string:
        disease = "Healthy Control"
    else:
        disease = "Unknown"

    if 'baseline' in char_string:
        time_point = "Baseline"
    elif 'month' in char_string or 'week' in char_string:
        time_point = "Follow-up"
    else:
        time_point = "Unspecified"

    sample_list.append(gsm_id)
    disease_list.append(disease)
    time_point_list.append(time_point)
    characteristics_list.append(char_string)

df_metadata = pd.DataFrame({
    'Sample_ID': sample_list,
    'Disease_Status': disease_list,
    'Cohort_Timepoint': time_point_list,
    'Raw_Characteristics': characteristics_list
})

os.makedirs(PROCESSED_DIR, exist_ok=True)
meta_output_path = os.path.join(PROCESSED_DIR, 'metadata_patient_registry.csv')
df_metadata.to_csv(meta_output_path, index=False)

print("\n[+] Target Metadata Map Engineered Successfully!")
print(f"-> Metadata File Saved to: {meta_output_path}")
print("\n[-] Active Cohort Group Breakdown Counts:")
print(df_metadata.groupby(['Disease_Status', 'Cohort_Timepoint']).size())

10-Jun-2026 18:26:08 DEBUG utils - Directory /content/drive/MyDrive/PD ISEF project/data/raw/ already exists. Skipping.
DEBUG:GEOparse:Directory /content/drive/MyDrive/PD ISEF project/data/raw/ already exists. Skipping.
10-Jun-2026 18:26:08 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
10-Jun-2026 18:26:08 INFO GEOparse - Parsing /content/drive/MyDrive/PD ISEF project/data/raw/GSE269775_family.soft.gz: 
INFO:GEOparse:Parsing /content/drive/MyDrive/PD ISEF project/data/raw/GSE269775_family.soft.gz: 


[-] Parsing GEO dataset object from repository...


10-Jun-2026 18:26:08 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
10-Jun-2026 18:26:08 DEBUG GEOparse - SERIES: GSE269775
DEBUG:GEOparse:SERIES: GSE269775
10-Jun-2026 18:26:08 DEBUG GEOparse - PLATFORM: GPL17301
DEBUG:GEOparse:PLATFORM: GPL17301
10-Jun-2026 18:26:08 DEBUG GEOparse - SAMPLE: GSM8326771
DEBUG:GEOparse:SAMPLE: GSM8326771
10-Jun-2026 18:26:09 DEBUG GEOparse - SAMPLE: GSM8326772
DEBUG:GEOparse:SAMPLE: GSM8326772
10-Jun-2026 18:26:09 DEBUG GEOparse - SAMPLE: GSM8326773
DEBUG:GEOparse:SAMPLE: GSM8326773
10-Jun-2026 18:26:09 DEBUG GEOparse - SAMPLE: GSM8326774
DEBUG:GEOparse:SAMPLE: GSM8326774
10-Jun-2026 18:26:09 DEBUG GEOparse - SAMPLE: GSM8326775
DEBUG:GEOparse:SAMPLE: GSM8326775
10-Jun-2026 18:26:09 DEBUG GEOparse - SAMPLE: GSM8326776
DEBUG:GEOparse:SAMPLE: GSM8326776
10-Jun-2026 18:26:09 DEBUG GEOparse - SAMPLE: GSM8326777
DEBUG:GEOparse:SAMPLE: GSM8326777
10-Jun-2026 18:26:09 DEBUG GEOparse - SAMPLE: GSM8326778
DEBUG:GEOparse:SAMPLE: GSM8326778
1

[-] Extracting clinical attributes from parsed GSM objects...

[+] Target Metadata Map Engineered Successfully!
-> Metadata File Saved to: /content/drive/MyDrive/PD ISEF project/data/processed/metadata_patient_registry.csv

[-] Active Cohort Group Breakdown Counts:
Disease_Status       Cohort_Timepoint
Healthy Control      Unspecified         50
Parkinson's Disease  Unspecified         50
dtype: int64


In [5]:
print("[-] Checking for embedded expression data table...")

try:
    df_expression = gse.pivot_samples('VALUE')

    if not df_expression.empty:
        print(f"[+] Success! Embedded expression matrix found.")
        print(f"-> Matrix Dimensions: {df_expression.shape} (MicroRNAs vs Samples)")
        print("\n[-] First 5 rows of the expression matrix:")
        print(df_expression.head())
    else:
        print("[!] Object parsed, but expression table values are empty.")
except Exception as e:
    print(f"[-] Native matrix table not attached to master soft file. Details: {e}")
    print("-> This means we will pull the raw count spreadsheets directly from the subseries next!")

[-] Checking for embedded expression data table...
[-] Native matrix table not attached to master soft file. Details: 'ID_REF'
-> This means we will pull the raw count spreadsheets directly from the subseries next!


In [6]:
PROJECT_PATH = '/content/drive/MyDrive/PD ISEF project'
RAW_DIR = os.path.join(PROJECT_PATH, 'data/raw/')
os.makedirs(RAW_DIR, exist_ok=True)

cohort_file_map = {
    "GSE269775": "GSE269775_2020_CountMatrix.xlsx",
    "GSE269776": "GSE269776_2021_CountMatrix.xlsx",
    "GSE269777": "GSE269777_2022_CountMatrix.xlsx",
    "GSE269779": "GSE269779_2023_CountMatrix.xlsx"
}

print("[-] Connecting to NCBI Production Server...")

for series, filename in cohort_file_map.items():
    destination_path = os.path.join(RAW_DIR, f"{series}_raw_counts.xlsx")

    download_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?acc={series}&format=file&file={filename}"

    print(f"\n[-] Streaming File Endpoint: {series}")

    if not os.path.exists(destination_path):
        try:
            print(f" -> Pulling {filename}...")
            urllib.request.urlretrieve(download_url, destination_path)
            print(f" [+] Success! Saved to local path: {destination_path}")

            size_mb = os.path.getsize(destination_path) / (1024 * 1024)
            print(f" -> Download verified. Size: {size_mb:.2f} MB")

        except Exception as e:
            print(f" [X] Primary URL failed for {series}: {e}")
            print(" -> Attempting alternative HTTP fallback address...")
            try:
                fallback_url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{series[:-3]}nnn/{series}/suppl/{filename}"
                urllib.request.urlretrieve(fallback_url, destination_path)
                print(f" [+] Fallback success! Saved to: {destination_path}")
            except Exception as fallback_err:
                print(f" [X] Critical Failure on both paths for {series}: {fallback_err}")
    else:
        print(f" [!] Verified local copy already present. Skipping.")

print("\n" + "="*50)
print("[+] NOTEBOOK 01 CONCLUDED SUCCESSFULLY")
print("="*50)

[-] Connecting to NCBI Production Server...

[-] Streaming File Endpoint: GSE269775
 [!] Verified local copy already present. Skipping.

[-] Streaming File Endpoint: GSE269776
 [!] Verified local copy already present. Skipping.

[-] Streaming File Endpoint: GSE269777
 [!] Verified local copy already present. Skipping.

[-] Streaming File Endpoint: GSE269779
 [!] Verified local copy already present. Skipping.

[+] NOTEBOOK 01 CONCLUDED SUCCESSFULLY
